# Задача 3. Рюкзак жадный

In [1]:
def knapsack_greedy(items, W):
    """items: список (weight, value). Жадный по value/weight."""
    items = sorted(items, key=lambda x: x[1]/x[0], reverse=True)
    total_value = 0
    total_weight = 0
    chosen = []
    for w, v in items:
        if total_weight + w <= W:
            chosen.append((w, v))
            total_weight += w
            total_value += v
    return chosen, total_value

# Задача 3. Рюкзак точный

In [3]:
def knapsack_bruteforce(items, W):
    """Точный перебор для n ≤ 20."""
    n = len(items)
    best_value = 0
    best_combo = []
    for mask in range(1 << n):
        w = v = 0
        combo = []
        for i in range(n):
            if mask & (1 << i):
                w += items[i][0]
                v += items[i][1]
                combo.append(items[i])
        if w <= W and v > best_value:
            best_value = v
            best_combo = combo
    return best_combo, best_value

In [4]:
# Контрпример: жадный проигрывает
items = [(6, 30), (5, 25), (5, 25)]
W = 10
print("Жадный:", knapsack_greedy(items, W))
# ([6,30], [5,25]?) — 6+5=11>10, берём только (6,30) → 30
print("Точный:", knapsack_bruteforce(items, W))
# (5,25)+(5,25) = 50

Жадный: ([(6, 30)], 30)
Точный: ([(5, 25), (5, 25)], 50)


# Раскраска графа

In [5]:
def graph_coloring(graph):
    """
    graph: dict {вершина: [соседи]}
    Возвращает: dict {вершина: цвет}
    """
    colors = {}
    for node in graph:
        used = {colors[nbr] for nbr in graph[node] if nbr in colors}
        color = 0
        while color in used:
            color += 1
        colors[node] = color
    return colors

In [6]:
# Тест
graph = {
    "A": ["B", "C"],
    "B": ["A", "C", "D"],
    "C": ["A", "B", "D"],
    "D": ["B", "C"],
}
print(graph_coloring(graph))
# {'A': 0, 'B': 1, 'C': 2, 'D': 0} — 3 цвета

{'A': 0, 'B': 1, 'C': 2, 'D': 0}


# Рекурсия и динамическое программирование

### Наивная рекурсия
Что происходит: при вызове fib(5) функция вызывает fib(4) и fib(3). Но fib(3) вызывается дважды: из fib(4) и из fib(5). С ростом n число повторных вызовов растёт экспоненциально.

In [7]:
def fib_naive(n):
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

Дерево вызовов для fib(5):

                    fib(5)
                  /        \
              fib(4)        fib(3)
             /     \        /     \
         fib(3)  fib(2)  fib(2)  fib(1)
         /   \    /   \   /   \
     fib(2) fib(1) ...

### Динамическое программирование
Запоминаем результаты уже посчитанных значений в словаре или массиве. Если значение уже вычислено — возвращаем его без повторного вычисления.

In [8]:
def fib_memo(n, memo=None):
    if memo is None:
        memo = {}
    if n <= 1:
        return n
    if n in memo:
        return memo[n]
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]

Что изменилось: каждый fib(k) вычисляется ровно один раз. Повторные вызовы берут готовый ответ из memo.

Сложность:

Время: O(n) — каждый индекс от 0 до n вычисляется один раз.

Память: O(n) — словарь + стек вызовов.

Плюсы: сохраняет структуру рекурсии, легко читается.
Минусы: рекурсия → риск переполнения стека для больших n (в Python по умолчанию ~1000).

### Табуляция (bottom-up DP) — O(n)
Идём от базы вверх. Сначала считаем F(0), F(1), потом F(2) и так далее до F(n). Храним массив значений.

In [9]:
def fib_tab(n):
    if n <= 1:
        return n
    dp = [0] * (n + 1)
    dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]

Сложность:

Время: O(n).

Память: O(n) — массив dp.

Плюсы: нет рекурсии — можно считать fib(10000).
Минусы: используем O(n) памяти, хотя нужны только два последних значения.

### Оптимизация памяти — O(1)

In [10]:
def fib_optimal(n):
    if n <= 1:
        return n
    prev, curr = 0, 1
    for _ in range(2, n + 1):
        prev, curr = curr, prev + curr
    return curr

Сложность:

Время: O(n).

Память: O(1) — только две переменные.

Это финальное решение (которое ожидают увидеть на собеседовании).

In [18]:
import time
for n in [10, 20, 30, 35, 37]:
    t = time.time()
    fib_naive(n)
    print(f"naive({n}): {time.time() - t:.4f} сек")

naive(10): 0.0000 сек
naive(20): 0.0013 сек
naive(30): 0.1112 сек
naive(35): 1.1357 сек
naive(37): 2.9686 сек


In [14]:
import time
for n in [10, 20, 30, 35, 50]:
    t = time.time()
    fib_memo(n)
    print(f"naive({n}): {time.time() - t:.4f} сек")

naive(10): 0.0000 сек
naive(20): 0.0000 сек
naive(30): 0.0000 сек
naive(35): 0.0000 сек
naive(50): 0.0000 сек


In [15]:
import time
for n in [10, 20, 30, 35, 50]:
    t = time.time()
    fib_optimal(n)
    print(f"naive({n}): {time.time() - t:.4f} сек")

naive(10): 0.0000 сек
naive(20): 0.0000 сек
naive(30): 0.0000 сек
naive(35): 0.0000 сек
naive(50): 0.0000 сек


## Для самостоятельной работы (оформляем в гитхаб на семинаре или дома)

### Climbing Stairs
Условие:
Вы поднимаетесь по лестнице из n ступенек. За один шаг можно подняться на 1 или 2 ступеньки. Сколько существует различных способов добраться до вершины?

Примеры:
n = 2 → 2 (1+1, 2)
n = 3 → 3 (1+1+1, 1+2, 2+1)

Подсказка:

Обозначим dp[i] — число способов дойти до ступеньки i.
На ступеньку i можно попасть с i-1 (шаг 1) или с i-2 (шаг 2).
Значит, dp[i] = dp[i-1] + dp[i-2].
База: dp[0] = 1, dp[1] = 1.

Это как числа Фибоначчи, только со сдвигом.
Требование: решить тремя способами — мемоизация, табуляция, O(1) по памяти.
Сложность: O(n) по времени, O(1) по памяти (для оптимального решения).

### House Robber
Условие:
Вы — грабитель, планирующий ограбить дома на улице. В каждом доме i лежит nums[i] денег. Единственное ограничение: нельзя грабить два соседних дома — сработает сигнализация. Найдите максимальную сумму, которую можно украсть.

nums = [1,2,3,1]   → 4 (грабим дома 0 и 2: 1 + 3)
nums = [2,7,9,3,1] → 12 (грабим дома 0, 2, 4: 2 + 9 + 1)

Подсказка:
Обозначим dp[i] — максимальная сумма, которую можно украсть из первых i домов.
На каждом шаге выбор: ограбить дом i (nums[i] + dp[i-2]) или пропустить его (dp[i-1]).
Формула: dp[i] = max(dp[i-1], dp[i-2] + nums[i]).
База: dp[0] = nums[0], dp[1] = max(nums[0], nums[1]).
Оптимизация: можно хранить только две последние переменные → O(1) памяти.
Сложность: O(n) по времени, O(1) по памяти.

На что обратить внимание:
Краевые случаи: пустой массив, один дом, два дома.
** Подумайте, как модифицировать решение, если дома стоят по кругу.